In [8]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../src/")))

ROOT_DIR = "../.."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [10]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

Stopping existing Spark context...
Previous Spark context stopped successfully


In [11]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

New Spark session created successfully


In [12]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [17]:
import pandas as pd

# alz_df_pandas = pd.read_pickle("alz_df_apr10_1355.pkl")
# cntrl_df_pandas = pd.read_pickle("cntrl_df_apr10_1355.pkl")


# alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr15_1033.pkl")
# cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr15_1033.pkl")

alz_df_pandas = pd.read_pickle("../features_alz_extra_features_Apr15_2135.pkl")
cntrl_df_pandas = pd.read_pickle("../features_cntrl_extra_features_Apr15_2135.pkl")


In [18]:
%%time
alz_df_spark = spark.createDataFrame(alz_df_pandas)
cntrl_df_spark = spark.createDataFrame(cntrl_df_pandas)

CPU times: user 6min 57s, sys: 4.06 s, total: 7min 1s
Wall time: 7min 7s


In [19]:
%%time
#just renaming things now that we understand the types and where things are coming from
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

CPU times: user 1e+03 ns, sys: 0 ns, total: 1e+03 ns
Wall time: 3.1 µs


In [21]:
alz_df.show(4)

25/04/18 11:51:02 WARN TaskSetManager: Stage 0 contains a task of very large size (69323 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+---------+-------+---------+--------+-----------+--------------------+----------+-----+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|        FeatureValue|table_type|label|
+---------+-------+---------+--------+-----------+--------------------+----------+-----+
|  sub-008|ep-1651|     NULL|    NULL|        Std|6.130968540674075E-5|     epoch|    1|
|  sub-017|ep-1479|       Pz|    NULL| TotalPower| 0.01123595517128706| electrode|    1|
|  sub-017|ep-2651|      Fp1|    NULL|TotalEnergy| 0.23016475141048431| electrode|    1|
|  sub-017| ep-149|       Cz|   Alpha|      Power|0.001706972136162...|      band|    1|
+---------+-------+---------+--------+-----------+--------------------+----------+-----+
only showing top 4 rows



In [22]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

# Data Processing

In [23]:
# union everything
full_df = alz_df.unionByName(cntrl_df)

In [24]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [25]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(16).persist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(16).persist()

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(16).persist()


In [26]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

25/04/18 11:59:01 WARN TaskSetManager: Stage 10 contains a task of very large size (57564 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [27]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()


25/04/18 11:59:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: double, C3_Beta_Power: double, C3_Delta_Power: double, C3_Theta_Power: double, C3_custom1_Power: double, C4_Alpha_Power: double, C4_Beta_Power: double, C4_Delta_Power: double, C4_Theta_Power: double, C4_custom1_Power: double, Cz_Alpha_Power: double, Cz_Beta_Power: double, Cz_Delta_Power: double, Cz_Theta_Power: double, Cz_custom1_Power: double, F3_Alpha_Power: double, F3_Beta_Power: double, F3_Delta_Power: double, F3_Theta_Power: double, F3_custom1_Power: double, F4_Alpha_Power: double, F4_Beta_Power: double, F4_Delta_Power: double, F4_Theta_Power: double, F4_custom1_Power: double, F7_Alpha_Power: double, F7_Beta_Power: double, F7_Delta_Power: double, F7_Theta_Power: double, F7_custom1_Power: double, F8_Alpha_Power: double, F8_Beta_Power: double, F8_Delta_Power: double, F8_Theta_Power: double, F8_custom1_Power: double, Fp1_Alpha_Power: double, Fp1_Beta_Power: double, Fp1_Delta_Power: double, Fp1_Theta_Power: doub

In [28]:
NUM_TEST_SUBJECTS_PER_GROUP = 2

# Get test subject IDs from full_df (which has .label)
alz_test_subjects = (
    full_df.filter("label == 1")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

cntrl_test_subjects = (
    full_df.filter("label == 0")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

test_subjects = alz_test_subjects + cntrl_test_subjects #this will be the firs 2 subjects of each group for reproduceablility

In [29]:
# Split into test and train sets
test_df = full_df.filter(col("SubjectID").isin(test_subjects))
train_df = full_df.filter(~col("SubjectID").isin(test_subjects))


In [30]:
import dimensionality_reduction
import importlib
importlib.reload(dimensionality_reduction)
from dimensionality_reduction import min_max_normalize
feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]
train_norm_df, test_norm_df = min_max_normalize(train_df, test_df, feature_cols)

In [31]:
pca_input_cols = feature_cols
from dimensionality_reduction import fit_pca_model

pca_model, k_val = fit_pca_model(train_norm_df, pca_input_cols, variance_target=0.95)

print(f"PCA model fitted with {k_val} components to capture 95% variance")

25/04/18 12:00:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/04/18 12:00:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
                                                                                

PCA model fitted with 22 components to capture 95% variance


In [32]:
pca_model.explainedVariance

DenseVector([0.5277, 0.1446, 0.0776, 0.0358, 0.0305, 0.0264, 0.0225, 0.0136, 0.0112, 0.01, 0.0078, 0.0071, 0.0058, 0.0047, 0.0042, 0.0039, 0.0037, 0.0034, 0.0029, 0.0028, 0.0026, 0.0024])

In [33]:
from dimensionality_reduction import apply_pca_model

train_df = apply_pca_model(train_norm_df, pca_input_cols, pca_model, k_val)
test_df = apply_pca_model(test_norm_df, pca_input_cols, pca_model, k_val)


# ML time

In [44]:
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()
spark.stop()

AttributeError: 'NoneType' object has no attribute 'setCallSite'

In [45]:
import numpy as np

# Convert Spark DenseVectors to regular 2D numpy arrays
X_train = np.array(train_pd["features"].tolist())
y_train = train_pd["label"].values

X_test = np.array(test_pd["features"].tolist())
y_test = test_pd["label"].values

In [46]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


X_train shape: (168652, 22)
y_train shape: (168652,)
X_test shape: (10176, 22)
y_test shape: (10176,)


In [50]:
%%time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once
scaler = MinMaxScaler(feature_range=(-1, 1))  # You can use (0, 1) if you prefer
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Define hyperparameter grids
# layer_configs = [(100,), (128,), (128, 64), (256, 128, 64)]
# activations = ['relu', 'tanh']
# alphas = [1e-4, 1e-3, 1e-2]
# early_stopping_options = [True, False]

layer_configs = [(256, 128, 64), (128, 64, 16), (64, 16, 8)]  # Focus on best performing architecture
activations = ['relu'] #, 'tanh']  # Focus on best performing activation
alphas = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 5e-1, 1.0]  # Expanded range of L2 regularization strength
early_stopping_options = [True, False]
max_iter = 30000


# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force loop over hyperparameter combinations
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}, max_iter={max_iter}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=max_iter,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation accuracy scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Find best fold for detailed summary
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Mean Accuracy: 0.8893
Std Deviation: 0.0042
All Fold Scores: [0.8811 0.8916 0.8913 0.8894 0.8929]

=== Best Fold Summary: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Train Accuracy: 0.9221
Test Accuracy: 0.8867
              precision    recall  f1-score   support

     Control       0.88      0.86      0.87     15123
 Alzheimer's       0.89      0.91      0.90     18607

    accuracy                           0.89     33730
   macro avg       0.89      0.88      0.89     33730
weighted avg       0.89      0.89      0.89     33730

⏱️ Duration: 100.5s

=== Cross-Validation: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=False, max_iter=30000 ===
Mean Accuracy: 0.8978
Std Deviation: 0.0027
All Fold Scores: [0.8945 0.9019 0.8987 0.8951 0.8988]

=== Best Fold Summary: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=False, max_iter=30000 ===


# Trying with pytorch

In [43]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once
scaler = MinMaxScaler(feature_range=(-1, 1))  # You can use (0, 1) if you prefer
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Define hyperparameter grids - focus on regularization
layer_configs = [(256, 128, 64), (128, 64, 16), (64, 16, 8)]  # Focus on best performing architecture
activations = ['relu', 'tanh']  # Focus on best performing activation
alphas = [1e-4, 1e-3, 1e-2, 1e-1]  # L2 regularization strengthe
arly_stopping_options = [True, False]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force loop over hyperparameter combinations
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:
                
                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,  # L2 regularization
                    early_stopping=early_stopping,
                    max_iter=20000,  # Double the iterations from 10000 to 20000
                    random_state=42,
                    solver='adam',
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation accuracy scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Find best fold for detailed summary
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


=== Cross-Validation: MLP (256, 128, 64), act=relu, alpha=0.0001, early_stop=True ===
Mean Accuracy: 0.8800
Std Deviation: 0.0219
All Fold Scores: [0.92 0.86 0.88 0.86 0.88]

=== Best Fold Summary: MLP (256, 128, 64), act=relu, alpha=0.0001, early_stop=True ===
Train Accuracy: 0.9200
Test Accuracy: 0.8800
              precision    recall  f1-score   support

     Control       0.88      0.88      0.88        50
 Alzheimer's       0.88      0.88      0.88        50

    accuracy                           0.88       100
   macro avg       0.88      0.88      0.88       100
weighted avg       0.88      0.88      0.88       100

⏱️ Duration: 0.8s

=== Cross-Validation: MLP (256, 128, 64), act=relu, alpha=0.0001, early_stop=False ===
Mean Accuracy: 0.8420
Std Deviation: 0.0214
All Fold Scores: [0.86 0.83 0.84 0.81 0.87]

=== Best Fold Summary: MLP (256, 128, 64), act=relu, alpha=0.0001, early_stop=False ===
Train Accuracy: 1.0000
Test Accuracy: 0.7900
              precision    recall  f1